In [ ]:
# Dependencies and imports - RUN THIS CELL FIRST
# Running this cell might take up to 30 seconds
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets
from datetime import datetime
from DS1104_interface_V2 import ThermalControlApparatus, CollectMeasurementData, AnimateScope
import control as ct

plant = ThermalControlApparatus() # Load thermal control apparatus interface
plant.start() # Start the apparatus (This can take up to 30 seconds)   

# Modeling: obtain and test a dynamic model of the system

In [ ]:
# Below we can define a model of the plant

num = [22.0]
den = [350.0, 1.0]
dead_time = 5.0
def create_transferFucntion(numerator, denominator):
    plant = ct.tf(numerator, denominator)
    n, d = ct.pade(dead_time, 3) # Create a Pade approximation of the dead time
    plant_delay = ct.tf(n, d)
    return plant*plant_delay

plant_model = create_transferFucntion(num, den)

In [ ]:
print(ct.tf2ss(plant_model))

 Press "Start Measuring" in D-Space Before continuing!

![alt text](start_measuring_dspace.png "Start Measurement Procedure")

### The scope now shows your model as the dotted line (---) besides the measured inputs

In [ ]:
# Signal Scope Cell
ani = None # Global variable to hold the animation object
plt.close("all") # Close all previous figures to avoid overlap

window_length = 20 # seconds
dt = 0.1 # seconds
def run_T_scope():
    global ani
    if ani is not None:
        ani.event_source.stop()  # Stop the existing animation

    plt.close("all")  # Close all previous figures to avoid overlap
    # Create two subplots: one for temperatures, one for inputs
    fig, (ax_temp, ax_input) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Temperature lines
    # Temperature lines
    line1, = ax_temp.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_1')
    line2, = ax_temp.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_2')
    line_model, = ax_temp.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_model', linestyle='--')
    
    # Input lines
    line3, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Heater')
    line4, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Fan')
    line5, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Vane')
    
    # Plot settings, feel free to change these how you see fit
    #Temperature plot settings
    ax_temp.legend()
    ax_temp.set_ylim(0, 100)
    ax_temp.set_title("Live Temperature Readings")
    ax_temp.set_ylabel("Temperature (°C)")
    ax_temp.grid(True)

    #Input signal plot settings
    ax_input.legend()
    ax_input.set_ylim(0, 12)
    ax_input.set_title("Supplied Inputs")
    ax_input.set_xlabel("Time (s)")
    ax_input.set_ylabel("Input Value (V)")
    ax_input.grid(True)

    # Use the AnimateScope signature
    state = AnimateScope([line1, line2, line_model], [line3, line4, line5], ax_temp, ax_input, window_length=window_length, dt=dt, plant=plant, plant_model=plant_model)
    ani = animation.FuncAnimation(fig,
                                 state.animate,
                                 init_func=state.init_plot,
                                 interval=100,
                                 blit=True,
                                 save_count=0)

    plt.tight_layout()
    plt.show()

run_T_scope()

### Now change the inputs, How does the model react?

In [ ]:
HeatSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Heater Power')
FanSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Fan Power')
VaneSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Vane Rotation')

widgets.interact(plant.write_heater, heater_input=HeatSlider);
widgets.interact(plant.write_fan, fan_input=FanSlider);
widgets.interact(plant.write_vane, vane_input=VaneSlider);

### Take measurements, these are stored in "C:\Thermal\Project_Thermal\ExperimentData"

In [ ]:
# Run this cell to collect the data, which enables the logging of data to a CSV file, 

data_collector = CollectMeasurementData(plant, fs=10.0, Exp_length_minutes=20.0)

### After you're done experimenting, always run the cool-down command cell before exiting the environment

In [ ]:
# cool down the system when done experimenting
plant.cooling_down()
plant.stop()